# **Milestone 1 Experiment — First-Pass Affine MSE Stitching**

*COMS 4705 - Natural Language Processing - Final Project*

**Authors:** Caroline Zhang (`cz2977@columbia.edu`), Siddharth Grover (`sg4792@columbia.edu`), Edwin Argudo (`eaa2193@columbia.edu`)

**GitHub Repo:** https://github.com/edwin-a14/coms-4705-project-cross-model-stitching

---

### Purpose
Run a reproducible, **model-free first-pass MSE stitching experiment** mapping frozen `Llama-3.2-3B` pre-RMSNorm activations ($d_L = 3072$) into `Qwen2.5-3B`'s pre-RMSNorm space ($d_Q = 2048$) using the verified Milestone 1 Phase 5 handoff artifacts (`handoff.json`, `affine_init.pt`, `qwen_frozen_head.pt`, and paired `train`/`validation` activation caches).

### Scope & Protocol Guardrails
* **No 3B LLM backbones are loaded.** Training and evaluation operate strictly on cached activations and Qwen's exported frozen RMSNorm + LM head.
* **Shared initialization:** Starts from the exact untrained `affine_init.pt` (`trained_steps = 0`, `seed = 42`) exported in Milestone 1 Phase 5 so future CE and KL runs begin from identical weights.
* **Held-out test split remains locked:** Only `train` (10,000 examples) and `validation` (4,869 examples) are accessed; `allow_test=False` is enforced throughout.
* **Outputs saved to Drive:** Best checkpoint, training history, baseline comparison metrics, a publication-ready training curve plot (`mse_training_curves.png`), and a LaTeX-ready table snippet for the Milestone 1 report.

## **1. Dependencies, Google Drive & Configuration**

In [ ]:
%pip install -q tqdm matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path
from tempfile import NamedTemporaryFile

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Verified Milestone 1 Phase 5 handoff path
HANDOFF_PATH = Path(
    "/content/drive/MyDrive/coms-4705-project/activations/lambada/"
    "7b6847a7f51353d0/train10000_validation4869_a938da615918/handoff/handoff.json"
)
EXPERIMENT_NAME = "milestone1_mse_first_pass"

# First-pass MSE training hyperparameters (un-tuned baseline configuration)
SEED = 42
BATCH_SIZE = 32          # Matches Milestone 1 Phase 5 handoff batch size
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
EARLY_STOP_PATIENCE = 5
MIN_DELTA = 1e-4         # Minimum validation MSE improvement to reset patience

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# On an L4 GPU, evaluating full validation logits takes < 1s/epoch; throttle on CPU
EVAL_PRED_EVERY = 1 if DEVICE == "cuda" else 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
print(f"Handoff path exists: {HANDOFF_PATH.is_file()}")

## **2. Milestone 1 Phase 5 Handoff Interface**

These functions mirror the exact interface defined in `milestone1.ipynb` (Phase 5) so that all 25 artifacts are SHA-256 verified before training begins, splits match their frozen manifests, and the shared untrained `affine_init.pt` checkpoint is loaded cleanly.

In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def load_handoff(path):
    """Verify the complete inventory in a fresh runtime before loading any weights."""
    path = Path(path)
    report = json.loads(path.read_text())
    if report.get("schema") != "cross-family-milestone2-v2":
        raise ValueError("Unsupported handoff schema")
    for key, item in report["artifacts"].items():
        if file_hash(item["path"]) != item["sha256"]:
            raise ValueError(f"Handoff artifact changed: {key}")
    return report


def read_split(cache_path, manifest_path, split, fingerprint, metadata, widths, vocab_sizes):
    """Check alignment without tokenizers; retain repeated official evaluation contexts."""
    path = Path(cache_path)
    checksum = json.loads(path.with_suffix(".sha256.json").read_text())["sha256"]
    if file_hash(path) != checksum:
        raise ValueError(f"Paired-cache checksum mismatch: {split}")
    cache = torch.load(path, map_location="cpu", weights_only=True)
    provenance = cache["provenance"]
    if object_hash(provenance) != fingerprint or provenance.get("split") != split:
        raise ValueError(f"Wrong cache fingerprint/split: {split}")
    for key in ("dataset", "dataset_revision", "models", "revisions", "max_tokens"):
        if provenance.get(key) != metadata[key]:
            raise ValueError(f"Wrong {split} provenance: {key}")
    if (
        cache.get("source_model") != metadata["models"]["llama"]
        or cache.get("target_model") != metadata["models"]["qwen"]
    ):
        raise ValueError(f"Wrong model identities: {split}")
    raw = Path(manifest_path).read_bytes()
    if hashlib.sha256(raw).hexdigest() != provenance["manifest_sha256"]:
        raise ValueError(f"Manifest checksum mismatch: {split}")
    rows = [json.loads(line) for line in raw.splitlines()]
    ids = [row["id"] for row in rows]
    if (
        not rows
        or len(rows) != provenance["n_examples"]
        or cache["ids"] != ids
        or len(ids) != len(set(ids))
    ):
        raise ValueError(f"Invalid row count/ID order: {split}")
    for row in rows:
        if row["split"] != split or not row["id"].startswith(split + "/"):
            raise ValueError(f"Invalid manifest ID/split: {split}")
        if not row["prefix"].strip() or not row["continuation"].strip():
            raise ValueError(f"Empty text: {row['id']}")
        for name in widths:
            if not 0 < row[f"{name}_length"] <= metadata["max_tokens"] or row[f"{name}_target_length"] < 1:
                raise ValueError(f"Invalid token lengths: {row['id']}")
    for name, width in widths.items():
        key = "target_token_ids" if name == "qwen" else "llama_target_token_ids"
        labels = cache[key]
        expected = torch.tensor([row[f"{name}_target_id"] for row in rows], dtype=torch.int64)
        if (
            labels.dtype != torch.int64
            or not torch.equal(labels, expected)
            or (labels < 0).any()
            or (labels >= vocab_sizes[name]).any()
        ):
            raise ValueError(f"Invalid {name} label alignment/range: {split}")
        for suffix in ("", "_post"):
            matrix = cache[f"h_{name}{suffix}"]
            if (
                matrix.shape != (len(rows), width)
                or not matrix.is_floating_point()
                or str(matrix.dtype) != provenance["extraction"]["compute_dtype"]
                or not torch.isfinite(matrix).all()
            ):
                raise ValueError(f"Invalid {name}{suffix} matrix: {split}")
    return rows, cache


class PairedActivations(Dataset):
    """Keep saved precision on CPU; convert only requested rows to float32."""
    def __init__(self, cache, rows, indices=None):
        self.indices = tuple(range(len(rows))) if indices is None else tuple(indices)
        self.x, self.y = cache["h_llama"], cache["h_qwen"]
        self.labels, self.ids = cache["target_token_ids"], tuple(cache["ids"])
        self.single = torch.tensor([r["qwen_target_length"] == 1 for r in rows])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        index = self.indices[index]
        return {
            "X": self.x[index].float(),
            "Y": self.y[index].float(),
            "labels": self.labels[index],
            "ids": self.ids[index],
            "single_token_mask": self.single[index],
        }


def make_loader(handoff, split, batch_size=32, seed=42, *, allow_test=False, test_view="disjoint", shuffle=None):
    """Only training shuffles by default. Test requires explicit opt-in."""
    if split not in ("train", "validation", "test") or batch_size <= 0:
        raise ValueError("Use a named split and a positive batch size")
    if split == "test" and not allow_test:
        raise ValueError("Test is reserved for final evaluation; set allow_test=True explicitly")
    item = handoff["splits"][split]
    for suffix, field in (("cache", "cache_path"), ("manifest", "manifest_path")):
        artifact = handoff["artifacts"][f"{split}_{suffix}"]
        if (
            Path(item[field]).resolve() != Path(artifact["path"]).resolve()
            or file_hash(item[field]) != artifact["sha256"]
        ):
            raise ValueError(f"{split} {suffix} differs from the verified handoff")
    rows, cache = read_split(
        item["cache_path"], item["manifest_path"], split,
        item["fingerprint"], handoff["metadata"],
        handoff["widths"], handoff["vocab_sizes"],
    )
    indices = None
    if split == "test":
        if test_view not in ("disjoint", "official"):
            raise ValueError("test_view must be disjoint or official")
        view = handoff["evaluation_views"][f"test_{test_view}"]
        excluded = set(view["excluded_ids"])
        if not excluded.issubset(set(cache["ids"])):
            raise ValueError("Test view references IDs absent from this cache")
        indices = [i for i, value in enumerate(cache["ids"]) if value not in excluded]
        selected_ids = [cache["ids"][i] for i in indices]
        if len(indices) != view["n_examples"] or object_hash(selected_ids) != view["ids_sha256"]:
            raise ValueError("Test view count/order differs from the handoff")
    dataset = PairedActivations(cache, rows, indices)
    do_shuffle = (split == "train") if shuffle is None else bool(shuffle)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=do_shuffle,
        generator=generator, num_workers=0, drop_last=False,
    )


def new_stitch(source_width=3072, target_width=2048, seed=42):
    """Seeded PyTorch Linear initialization, with no change to the caller's RNG."""
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(seed)
        return nn.Linear(source_width, target_width, bias=True, device="cpu", dtype=torch.float32)


def load_stitch(handoff, device="cpu"):
    item = handoff["artifacts"]["affine_initialization"]
    if file_hash(item["path"]) != item["sha256"]:
        raise ValueError("Affine initialization changed after handoff")
    checkpoint = torch.load(item["path"], map_location="cpu", weights_only=True)
    if checkpoint["trained_steps"] != 0:
        raise ValueError("Expected an untrained initialization")
    config = handoff["affine"]
    stitch = new_stitch(config["source_width"], config["target_width"], config["seed"])
    stitch.load_state_dict(checkpoint["state_dict"])
    return stitch.to(device)


class FrozenQwenHead(nn.Module):
    """Frozen buffers preserve exported precision while gradients flow to inputs."""
    def __init__(self, state):
        super().__init__()
        self.eps = state["norm_eps"]
        self.register_buffer("norm_weight", state["norm_weight"])
        self.register_buffer("lm_head_weight", state["lm_head_weight"])

    def forward(self, hidden):
        hidden = hidden.to(self.norm_weight.dtype)
        normalized = hidden.float() * torch.rsqrt(hidden.float().square().mean(-1, keepdim=True) + self.eps)
        normalized = normalized.to(hidden.dtype) * self.norm_weight
        return F.linear(normalized, self.lm_head_weight)


def load_qwen_head(handoff, device="cpu"):
    item = handoff["artifacts"]["qwen_head"]
    if file_hash(item["path"]) != item["sha256"]:
        raise ValueError("Qwen head changed after handoff")
    state = torch.load(item["path"], map_location="cpu", weights_only=True)
    if (
        state["provenance"]["run_hash"] != handoff["run_hash"]
        or state["provenance"]["model"] != handoff["metadata"]["models"]["qwen"]
        or state["provenance"]["revision"] != handoff["metadata"]["revisions"]["qwen"]
    ):
        raise ValueError("Qwen head provenance differs from the handoff")
    return FrozenQwenHead(state).eval().requires_grad_(False).to(device)


def objective_loss(stitch, batch, objective, head=None, temperature=1.0):
    """One objective per experiment; calling this does not perform an optimizer step."""
    if objective not in ("mse", "ce", "kl"):
        raise ValueError("Select mse, ce, or kl")
    device = next(stitch.parameters()).device
    predicted = stitch(batch["X"].to(device=device, dtype=torch.float32))
    if objective == "mse":
        return F.mse_loss(predicted, batch["Y"].to(device=device, dtype=torch.float32))
    if head is None:
        raise ValueError("CE/KL require the exported frozen Qwen head on the stitch device")
    if any(p.requires_grad for p in head.parameters()):
        raise ValueError("The output head must be frozen")
    logits = head(predicted).float()
    if objective == "ce":
        return F.cross_entropy(logits, batch["labels"].to(device))
    if not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("KL temperature must be positive and finite")
    with torch.no_grad():
        teacher_logits = head(batch["Y"].to(device=device, dtype=torch.float32)).float()
        teacher_logp = F.log_softmax(teacher_logits / temperature, dim=-1)
    student_logp = F.log_softmax(logits / temperature, dim=-1)
    return F.kl_div(student_logp, teacher_logp, log_target=True, reduction="batchmean") * temperature**2


def show_table(title, columns, rows):
    def fmt(val):
        if isinstance(val, float):
            return f"{val:.3f}"
        return str(val).replace("|", "\|").replace("\n", " ")
    lines = [
        f"### {title}", "",
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    lines += ["| " + " | ".join(map(fmt, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

## **3. Load Verified Handoff, DataLoaders & Frozen Qwen Head**

In [ ]:
handoff = load_handoff(HANDOFF_PATH)
print(f"Verified handoff schema: {handoff['schema']} ({len(handoff['artifacts'])} artifacts)")

train_loader = make_loader(handoff, "train", batch_size=BATCH_SIZE, seed=SEED)
train_eval_loader = make_loader(handoff, "train", batch_size=EVAL_BATCH_SIZE, seed=SEED, shuffle=False)
val_loader = make_loader(handoff, "validation", batch_size=EVAL_BATCH_SIZE, seed=SEED)

stitch = load_stitch(handoff, device=DEVICE)
qwen_head = load_qwen_head(handoff, device=DEVICE)

# Load saved train-only constant mean for geometric baseline reference
mean_payload = torch.load(
    handoff["artifacts"]["training_mean"]["path"], map_location="cpu", weights_only=True
)
qwen_train_mean = mean_payload["mean"].float().to(DEVICE)

n_params = sum(p.numel() for p in stitch.parameters() if p.requires_grad)
print(f"Train examples:      {len(train_loader.dataset):,}")
print(f"Validation examples: {len(val_loader.dataset):,}")
print(f"Trainable params:    {n_params:,} ({stitch.in_features} -> {stitch.out_features} + bias)")
print(f"Qwen head frozen:    {all(not p.requires_grad for p in qwen_head.parameters())}")

## **4. Evaluation Functions**

Evaluates any candidate Qwen pre-RMSNorm representation (either `stitch(X)` or the constant `qwen_train_mean`) across all proposal metrics:
* **Geometric alignment:** Activation MSE (mean squared error per coordinate), mean cosine similarity, and relative $L_2$ error.
* **Next-token prediction:** Cross-entropy (CE), top-1 accuracy, and single-token target exact-match accuracy in Qwen's vocabulary.
* **Functional compatibility:** Top-1 prediction agreement and output KL divergence $D_{\text{KL}}(P_{\text{Qwen}} \| P_{\text{stitch}})$ relative to native Qwen.

In [ ]:
@torch.no_grad()
def evaluate_geometric(stitch_model, loader, device):
    """Fast geometric-only pass (no LM head projection) for per-epoch tracking."""
    stitch_model.eval()
    n_total = 0
    mse_sum = 0.0
    cos_sum = 0.0
    rel_l2_sum = 0.0
    pred_norm_sum = 0.0
    target_norm_sum = 0.0

    for batch in loader:
        x = batch["X"].to(device=device, dtype=torch.float32)
        y = batch["Y"].to(device=device, dtype=torch.float32)
        pred = stitch_model(x)
        bsz, width = y.shape

        mse_sum += F.mse_loss(pred, y, reduction="sum").item()
        cos_sum += F.cosine_similarity(pred, y, dim=-1).sum().item()
        rel_l2_sum += ((pred - y).norm(dim=-1) / y.norm(dim=-1).clamp_min(1e-12)).sum().item()
        pred_norm_sum += pred.norm(dim=-1).sum().item()
        target_norm_sum += y.norm(dim=-1).sum().item()
        n_total += bsz

    return {
        "n_examples": n_total,
        "mse": mse_sum / (n_total * width),
        "cosine_similarity": cos_sum / n_total,
        "mean_rel_l2": rel_l2_sum / n_total,
        "pred_mean_l2_norm": pred_norm_sum / n_total,
        "target_mean_l2_norm": target_norm_sum / n_total,
    }


@torch.no_grad()
def evaluate_full(predictor_fn, loader, head, device, desc="eval"):
    """Full geometric + predictive + native-Qwen compatibility evaluation."""
    n_total = 0
    width = None
    mse_sum = 0.0
    cos_sum = 0.0
    rel_l2_sum = 0.0
    pred_norm_sum = 0.0

    ce_sum = 0.0
    correct = 0
    single_n = 0
    single_correct = 0
    agree_qwen = 0
    kl_sum = 0.0

    for batch in tqdm(loader, desc=desc, leave=False):
        x = batch["X"].to(device=device, dtype=torch.float32)
        y = batch["Y"].to(device=device, dtype=torch.float32)
        labels = batch["labels"].to(device)
        single_mask = batch["single_token_mask"].to(device)

        pred_h = predictor_fn(x, y)
        bsz, width = y.shape

        # Geometric metrics
        mse_sum += F.mse_loss(pred_h, y, reduction="sum").item()
        cos_sum += F.cosine_similarity(pred_h, y, dim=-1).sum().item()
        rel_l2_sum += ((pred_h - y).norm(dim=-1) / y.norm(dim=-1).clamp_min(1e-12)).sum().item()
        pred_norm_sum += pred_h.norm(dim=-1).sum().item()

        # Predictive & compatibility metrics via frozen Qwen head
        pred_logits = head(pred_h).float()
        native_logits = head(y).float()
        if not torch.isfinite(pred_logits).all():
            raise ValueError("Nonfinite stitched logits encountered during evaluation")

        preds = pred_logits.argmax(dim=-1)
        native_preds = native_logits.argmax(dim=-1)
        hits = preds == labels

        ce_sum += F.cross_entropy(pred_logits, labels, reduction="sum").item()
        correct += hits.sum().item()
        single_n += single_mask.sum().item()
        single_correct += hits[single_mask].sum().item()
        agree_qwen += (preds == native_preds).sum().item()

        native_logp = F.log_softmax(native_logits, dim=-1)
        pred_logp = F.log_softmax(pred_logits, dim=-1)
        kl_sum += (native_logp.exp() * (native_logp - pred_logp)).sum().item()

        n_total += bsz

    return {
        "n_examples": n_total,
        "mse": mse_sum / (n_total * width),
        "cosine_similarity": cos_sum / n_total,
        "mean_rel_l2": rel_l2_sum / n_total,
        "pred_mean_l2_norm": pred_norm_sum / n_total,
        "next_token_ce": ce_sum / n_total,
        "next_token_accuracy": correct / n_total,
        "single_token_count": single_n,
        "single_token_accuracy": (single_correct / single_n) if single_n else None,
        "agreement_with_native_qwen": agree_qwen / n_total,
        "kl_native_qwen_to_prediction": kl_sum / n_total,
    }

## **5. Train First-Pass MSE Affine Stitch**

Trains `stitch = nn.Linear(3072, 2048)` from the shared `affine_init.pt` using `AdamW` (`lr=1e-3`, `weight_decay=1e-4`), minimizing `objective_loss(stitch, batch, "mse")`. Tracks training and validation MSE every epoch, checkpoints the lowest validation-MSE weights, and stops early if validation MSE stops improving for `EARLY_STOP_PATIENCE` epochs.

In [ ]:
# Baseline check at epoch 0 (before any gradient updates)
init_train_geo = evaluate_geometric(stitch, train_eval_loader, DEVICE)
init_val_geo = evaluate_geometric(stitch, val_loader, DEVICE)
print(
    f"Epoch  0 (untrained) | "
    f"train MSE: {init_train_geo['mse']:.4f} (cos={init_train_geo['cosine_similarity']:.4f}) | "
    f"val MSE: {init_val_geo['mse']:.4f} (cos={init_val_geo['cosine_similarity']:.4f})"
)

optimizer = torch.optim.AdamW(
    stitch.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

history = []
best_val_mse = float("inf")
best_epoch = 0
best_state_dict = None
epochs_without_improvement = 0
total_steps = 0

t_start = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    stitch.train()
    epoch_loss_sum = 0.0
    epoch_examples = 0

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = objective_loss(stitch, batch, "mse")
        if not torch.isfinite(loss):
            raise ValueError(f"Nonfinite training MSE at epoch {epoch}, step {total_steps}")
        loss.backward()
        optimizer.step()

        bsz = batch["X"].shape[0]
        epoch_loss_sum += loss.item() * bsz
        epoch_examples += bsz
        total_steps += 1

    train_mse_epoch = epoch_loss_sum / epoch_examples
    val_geo = evaluate_geometric(stitch, val_loader, DEVICE)

    record = {
        "epoch": epoch,
        "steps": total_steps,
        "train_mse": train_mse_epoch,
        "val_mse": val_geo["mse"],
        "val_cosine": val_geo["cosine_similarity"],
        "val_rel_l2": val_geo["mean_rel_l2"],
        "val_pred_norm": val_geo["pred_mean_l2_norm"],
        "val_ce": None,
        "val_top1": None,
        "val_kl": None,
    }

    if epoch % EVAL_PRED_EVERY == 0 or epoch == 1:
        stitch.eval()
        val_pred_metrics = evaluate_full(
            lambda x, y: stitch(x), val_loader, qwen_head, DEVICE, desc=f"val pred ep {epoch}"
        )
        record["val_ce"] = val_pred_metrics["next_token_ce"]
        record["val_top1"] = val_pred_metrics["next_token_accuracy"]
        record["val_kl"] = val_pred_metrics["kl_native_qwen_to_prediction"]

    history.append(record)

    improved = val_geo["mse"] < (best_val_mse - MIN_DELTA)
    if improved:
        best_val_mse = val_geo["mse"]
        best_epoch = epoch
        best_state_dict = {k: v.detach().cpu().clone() for k, v in stitch.state_dict().items()}
        epochs_without_improvement = 0
        marker = " *"
    else:
        epochs_without_improvement += 1
        marker = ""

    pred_str = (
        f" | val CE: {record['val_ce']:.3f} | val Top-1: {record['val_top1']:.3f}"
        if record["val_ce"] is not None
        else ""
    )
    print(
        f"Epoch {epoch:2d}/{MAX_EPOCHS} | "
        f"train MSE: {train_mse_epoch:.4f} | "
        f"val MSE: {val_geo['mse']:.4f} (cos={val_geo['cosine_similarity']:.4f})"
        f"{pred_str}{marker}"
    )

    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch} (best validation MSE at epoch {best_epoch}).")
        break

train_elapsed_sec = time.perf_counter() - t_start
print(f"\nTraining complete in {train_elapsed_sec:.1f}s ({total_steps:,} steps). Best epoch: {best_epoch} (val MSE = {best_val_mse:.4f}).")

# Restore best validation checkpoint
stitch.load_state_dict({k: v.to(DEVICE) for k, v in best_state_dict.items()})
stitch.eval()

## **6. Full Evaluation & Baseline Comparison**

Evaluates the best MSE checkpoint on both `train` and `validation` splits and compares it side-by-side against the verified Milestone 1 baselines (`native_qwen`, `native_llama`, and `constant_qwen_train_mean`).

In [ ]:
mse_train_metrics = evaluate_full(
    lambda x, y: stitch(x), train_eval_loader, qwen_head, DEVICE, desc="MSE stitch (train)"
)
mse_val_metrics = evaluate_full(
    lambda x, y: stitch(x), val_loader, qwen_head, DEVICE, desc="MSE stitch (val)"
)

# Compute geometric metrics for the constant-activation baseline so all Qwen-space rows are comparable
const_train_geo = evaluate_geometric(
    nn.Identity(),
    [{"X": qwen_train_mean.expand(b["Y"].shape[0], -1), "Y": b["Y"]} for b in train_eval_loader],
    DEVICE,
)
const_val_geo = evaluate_geometric(
    nn.Identity(),
    [{"X": qwen_train_mean.expand(b["Y"].shape[0], -1), "Y": b["Y"]} for b in val_loader],
    DEVICE,
)

# Pull verified Phase 3 baseline metrics from handoff.json
baselines_by_key = {
    (row["split"], row["baseline"]): row for row in handoff["baseline_results"]
}

comparison_rows = []
for split, mse_res, const_geo in (
    ("train", mse_train_metrics, const_train_geo),
    ("validation", mse_val_metrics, const_val_geo),
):
    nq = baselines_by_key[(split, "native_qwen")]
    nl = baselines_by_key[(split, "native_llama")]
    cm = baselines_by_key[(split, "constant_qwen_train_mean")]

    comparison_rows.extend([
        [
            split, "Native Qwen (ceiling)", nq["n_examples"],
            0.000, 1.000, nq["next_token_ce"], nq["next_token_accuracy"],
            nq["single_token_accuracy"], nq["agreement_with_native_qwen"],
            nq["kl_native_qwen_to_prediction"],
        ],
        [
            split, "Native Llama (source ref)", nl["n_examples"],
            "—", "—", nl["next_token_ce"], nl["next_token_accuracy"],
            nl["single_token_accuracy"], "—", "—",
        ],
        [
            split, "Affine Stitch — MSE (first pass)", mse_res["n_examples"],
            mse_res["mse"], mse_res["cosine_similarity"],
            mse_res["next_token_ce"], mse_res["next_token_accuracy"],
            mse_res["single_token_accuracy"], mse_res["agreement_with_native_qwen"],
            mse_res["kl_native_qwen_to_prediction"],
        ],
        [
            split, "Constant Qwen Train Mean", cm["n_examples"],
            const_geo["mse"], const_geo["cosine_similarity"],
            cm["next_token_ce"], cm["next_token_accuracy"],
            cm["single_token_accuracy"], cm["agreement_with_native_qwen"],
            cm["kl_native_qwen_to_prediction"],
        ],
    ])

show_table(
    "Milestone 1 Preliminary Results — First-Pass MSE Stitch vs. Baselines",
    [
        "Split", "Model", "N", "Act. MSE", "Cosine",
        "Next-Tok CE", "Top-1 Acc", "1-Tok Acc", "Agree w/ Qwen", "KL(Qwen || Pred)",
    ],
    comparison_rows,
)

# Relative recovery of native Qwen performance on validation
val_nq = baselines_by_key[("validation", "native_qwen")]
val_cm = baselines_by_key[("validation", "constant_qwen_train_mean")]
mse_reduction_pct = 100.0 * (1.0 - mse_val_metrics["mse"] / const_val_geo["mse"])
top1_recovery_pct = 100.0 * (mse_val_metrics["next_token_accuracy"] / val_nq["next_token_accuracy"])
print(f"Validation activation MSE reduction vs. constant mean: {mse_reduction_pct:.1f}%")
print(f"Validation Top-1 accuracy recovery of native Qwen:     {top1_recovery_pct:.1f}%")
print(
    f"Validation mean L2 norm — predicted: {mse_val_metrics['pred_mean_l2_norm']:.1f} "
    f"vs. native Qwen: {init_val_geo['target_mean_l2_norm']:.1f}"
)

## **7. Plot Training Curves & Save Artifacts to Google Drive**

Saves the best MSE checkpoint (`best_mse_stitch.pt`), metrics/history JSON (`mse_results.json`), and a clean two-panel figure (`mse_training_curves.png`) ready to embed in the Milestone 1 Overleaf report.

In [ ]:
def atomic_save(path, payload, is_tensor=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with NamedTemporaryFile(dir=path.parent, delete=False) as handle:
        tmp = Path(handle.name)
    try:
        if is_tensor:
            torch.save(payload, tmp)
        else:
            tmp.write_text(payload, encoding="utf-8")
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)


exp_dir = Path(handoff["parent_cache_run_dir"]) / "experiments" / EXPERIMENT_NAME
exp_dir.mkdir(parents=True, exist_ok=True)

# Plot training curves
epochs = [h["epoch"] for h in history]
train_mses = [h["train_mse"] for h in history]
val_mses = [h["val_mse"] for h in history]
val_cos = [h["val_cosine"] for h in history]
pred_epochs = [h["epoch"] for h in history if h["val_top1"] is not None]
val_top1s = [h["val_top1"] for h in history if h["val_top1"] is not None]
val_ces = [h["val_ce"] for h in history if h["val_ce"] is not None]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Panel A: Activation MSE
axes[0].plot(epochs, train_mses, marker="o", markersize=3.5, label="Train MSE")
axes[0].plot(epochs, val_mses, marker="s", markersize=3.5, label="Validation MSE")
axes[0].axhline(
    const_val_geo["mse"], color="gray", linestyle="--", linewidth=1.2,
    label=f"Constant Mean Val MSE ({const_val_geo['mse']:.1f})",
)
axes[0].axvline(best_epoch, color="tab:green", linestyle=":", linewidth=1.2, label=f"Best Epoch ({best_epoch})")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Activation MSE (pre-RMSNorm)")
axes[0].set_title("Affine MSE Stitching — Reconstruction Loss")
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8.5)

# Panel B: Validation Next-Token Top-1 Accuracy & Cosine Similarity
ax_r = axes[1].twinx()
l1 = axes[1].plot(pred_epochs, val_top1s, color="tab:blue", marker="o", markersize=3.5, label="Val Top-1 Accuracy")
l2 = axes[1].axhline(
    val_nq["next_token_accuracy"], color="tab:blue", linestyle="--", alpha=0.6,
    label=f"Native Qwen Val Top-1 ({val_nq['next_token_accuracy']:.3f})",
)
l3 = ax_r.plot(epochs, val_cos, color="tab:orange", marker="^", markersize=3.5, label="Val Cosine Similarity")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Top-1 Accuracy", color="tab:blue")
ax_r.set_ylabel("Validation Activation Cosine Similarity", color="tab:orange")
axes[1].set_ylim(0.0, max(0.70, val_nq["next_token_accuracy"] + 0.08))
ax_r.set_ylim(0.0, 1.02)
axes[1].set_title("Validation Predictive Accuracy & Geometric Alignment")
axes[1].grid(alpha=0.3)
lines_all = l1 + [l2] + l3
axes[1].legend(lines_all, [l.get_label() for l in lines_all], loc="lower right", fontsize=8.5)

plt.tight_layout()
fig_path = exp_dir / "mse_training_curves.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

# Save best checkpoint & complete JSON record
ckpt_path = exp_dir / "best_mse_stitch.pt"
atomic_save(
    ckpt_path,
    {
        "state_dict": best_state_dict,
        "objective": "mse",
        "best_epoch": best_epoch,
        "best_val_mse": best_val_mse,
        "total_steps": total_steps,
        "handoff_path": str(HANDOFF_PATH),
        "handoff_run_hash": handoff["run_hash"],
        "hyperparameters": {
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "optimizer": "AdamW",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "max_epochs": MAX_EPOCHS,
            "early_stop_patience": EARLY_STOP_PATIENCE,
            "min_delta": MIN_DELTA,
        },
    },
    is_tensor=True,
)

results_payload = {
    "experiment": EXPERIMENT_NAME,
    "objective": "mse",
    "device": DEVICE,
    "torch_version": str(torch.__version__),
    "handoff_path": str(HANDOFF_PATH),
    "handoff_run_hash": handoff["run_hash"],
    "training_seconds": round(train_elapsed_sec, 2),
    "best_epoch": best_epoch,
    "total_epochs_run": len(history),
    "total_steps": total_steps,
    "hyperparameters": {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_epochs": MAX_EPOCHS,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "min_delta": MIN_DELTA,
    },
    "metrics": {
        "train": mse_train_metrics,
        "validation": mse_val_metrics,
        "constant_mean_geometric": {
            "train": const_train_geo,
            "validation": const_val_geo,
        },
    },
    "history": history,
    "test_evaluated": False,
}
results_path = exp_dir / "mse_results.json"
atomic_save(results_path, json.dumps(results_payload, indent=2, allow_nan=False) + "\n")

print("Saved checkpoint:", ckpt_path)
print("Saved results:   ", results_path)
print("Saved figure:    ", fig_path)

## **8. LaTeX Table Snippet (Validation Split) for Milestone 1 Report**

In [ ]:
val_nq = baselines_by_key[("validation", "native_qwen")]
val_nl = baselines_by_key[("validation", "native_llama")]
val_cm = baselines_by_key[("validation", "constant_qwen_train_mean")]
vm = mse_val_metrics

latex_table = f"""\\begin{{table}}[h]
\\centering
\\small
\\begin{{tabular}}{{lrrrrrrr}}
\\toprule
Model (Val, $N=4,869$) & Act. MSE & Cos & CE & Top-1 & 1-Tok Acc & Agree & KL \\\\
\\midrule
Native Qwen (ceiling) & 0.000 & 1.000 & {val_nq['next_token_ce']:.3f} & {val_nq['next_token_accuracy']:.3f} & {val_nq['single_token_accuracy']:.3f} & 1.000 & 0.000 \\\\
Native Llama (source ref) & -- & -- & {val_nl['next_token_ce']:.3f} & {val_nl['next_token_accuracy']:.3f} & {val_nl['single_token_accuracy']:.3f} & -- & -- \\\\
Constant Qwen Train Mean & {const_val_geo['mse']:.2f} & {const_val_geo['cosine_similarity']:.3f} & {val_cm['next_token_ce']:.3f} & {val_cm['next_token_accuracy']:.3f} & {val_cm['single_token_accuracy']:.3f} & {val_cm['agreement_with_native_qwen']:.3f} & {val_cm['kl_native_qwen_to_prediction']:.3f} \\\\
\\midrule
Affine Stitch -- MSE (first pass) & {vm['mse']:.2f} & {vm['cosine_similarity']:.3f} & {vm['next_token_ce']:.3f} & {vm['next_token_accuracy']:.3f} & {vm['single_token_accuracy']:.3f} & {vm['agreement_with_native_qwen']:.3f} & {vm['kl_native_qwen_to_prediction']:.3f} \\\\
\\bottomrule
\\end{{tabular}}
\\caption{{First-pass MSE affine stitching vs. baselines on LAMBADA validation ($N=4,869$; 1-Tok $N=2,788$ Qwen / $2,804$ Llama). Llama metrics use Llama's vocabulary.}}
\\label{{tab:m1_mse_results}}
\\end{{table}}"""

print(latex_table)